# Provenance Agent - Workflow Demo

The demo notebook for the provenance agent. 

Three parts:

1. **Software** - how a notebook's imported libraries become a
   citation-metadata DataFrame.
2. **Data** - how a notebook's used datasets are detected  become a
   citation-metadata DataFrame.
3. **The agent layer** - the three ways to invoke both workflows.

**Neither workflow returns citations** Each *injects a cell* into the target notebook, and the citations are that cell's output when you run it. Dataset retrieval calls `get_bibtex()` on a
LiPD object living in the kernel, which this code cannot reach from outside,
so it writes the retrieval code into your notebook instead. 

These functions modify the current notebook by default.

## Setup

The package is installed (`pip install -e ".[dev,google]"`), so everything
imports by name from anywhere. Reference the README for help.

Parts 1 and 2 need no API key. Only `agent.run` and
`%provenance` in Part 3 call a model.

In [ ]:
#import everything manually without python magic commands
from provenance_agent.notebook_io import (
    parse_notebook,
    extract_libraries,
    strip_ipython_directives,
    validate_libraries,
)
from provenance_agent.citations import collect_library_entries
from provenance_agent.dataset_detection import (
    detect_datasets,
    detect_datasets_with_diagnostics,
)
from provenance_agent.data import (
    build_retrieval_cell,
    filter_datasets,
    inject_retrieval_cells,
    generate_data_workflow,
)

---

# Part 1 - Software building blocks

The software library detection uses static AST parsing. It works on a notebook that has never
been run.

## 1.1 `parse_notebook` - scan a full notebook

Takes a path to a `.ipynb` and returns the sorted list of library
names it imports. Passing nothing auto-detects the current notebook, which
requires `ipynbname` and a running kernel.

In [ ]:
parse_notebook('../examples/C02_b_DA_with_individual_seasonality.ipynb')

## 1.2 `extract_libraries` - parse a snippet

Works on a  string rather than a file. It handles both `import X` and
`from X.Y import Z`, and always returns the top-level package
(`matplotlib.pyplot` becomes `matplotlib`).

In [ ]:
code_snippet = """
import numpy as np
import pandas as pd
from matplotlib.pyplot import plt
"""
print(extract_libraries(code_snippet))

## 1.3 `strip_ipython_directives` - make a cell parseable

Notebooks may have `%matplotlib inline` and `!pip install ...`,
which are not valid Python and would make `ast.parse` fail. These are stripped
before parsing. 

In [ ]:
raw = """
%matplotlib inline
!pip install pyleoclim
import pyleoclim as pyleo
"""
print(strip_ipython_directives(raw))

## 1.4 Complete software pipeline: parse, validate, collect

1. `parse_notebook` extracts the imported libraries.
2. `validate_libraries` splits a requested subset into found and not-found.
3. `collect_library_entries` merges the matching BibTeX from the packaged
   `Citations/` data into one DataFrame, deduped by DOI.

A library with no entry
becomes a `note` row rather than disappearing.

In [ ]:
libraries = parse_notebook('paleoPCAlite.ipynb')
print('imported:', libraries)

found, not_found = validate_libraries(['pandas', 'numpy', 'fake_lib'], libraries)
print('found:    ', found)
print('not found:', not_found)

In [ ]:
entries = collect_library_entries(found)
print(f'{len(entries)} citation entries')
entries

## 1.5 Against the real corpus

`notebooks/examples/` serves as an example. Running the parser over it
tests against real scientific notebooks.

In [ ]:
corpus = [
    '../examples/C02_b_DA_with_individual_seasonality.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/CMIP6_LMR.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/data_from_esm_cloudcat.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/spatial_snapshots_xarray_bonuses.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/VICS_dashboard.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/widget_primer.ipynb',
]

for path in corpus:
    print(f'{path.split("/")[-1]:<55} {parse_notebook(path)}')

---

# Part 2 - Data 

The data side is also a **deterministic** AST and data-flow analysis. 

## 2.1 `detect_datasets` - which datasets were actually used

Returns `[variable, tool]` pairs. A source is reported only when its lineage
reaches a recognized analysis call, or when it produces a live terminal 
DataFrame. **A dataset that is loaded and then abandoned is not reported** 
surprises people most.

Recognized sources are PyLiPD, PyleoTUPS, and LiPDGraph, plus `xarray` and
`pandas` loaders; although `xarray` and `pandas` wouldn't have citations.

In [ ]:
detect_datasets('paleoPCAlite.ipynb')



`detect_datasets_with_diagnostics` returns the same pairs plus an explanation for
every analysis call whose data it could not trace back to a source.

An empty result with **no** warnings means no analysis calls were found at all.
An empty result **with** warnings means analysis was found but could not be
connected to a recognized loader. 

## 2.2 `build_retrieval_cell` 

Each source gets a retrieval block that reuses the object already loaded in your
kernel. Supports PyLiPD, PyleoTUPS, and LiPDGraph

In [ ]:
for variable, tool in [('D', 'PyLiPD'), ('ds', 'PyleoTUPS'), ('filtered_df2', 'LiPDGraph')]:
    print(f'# --- {tool} ---')
    print(build_retrieval_cell(variable, tool))
    print()

## 2.3 `filter_datasets`

`filter_datasets` narrows the detector's `[notebook variable, tool]` pairs by tool type before the citation cell is created. 

In [ ]:
sample = [['D', 'PyLiPD'], ['ds', 'PyleoTUPS'], ['filtered_df2', 'LiPDGraph']]
print('all:      ', filter_datasets(sample))
print('LiPDGraph:', filter_datasets(sample, tool='LiPDGraph'))

## 2.4 `inject_retrieval_cells` 

**Appends one cell for every source (software or data)**. Re-running a workflow
replaces its own cell.

In [ ]:
import nbformat

demo = nbformat.v4.new_notebook()
inject_retrieval_cells(demo, [['filtered_df2', 'LiPDGraph']])
print(demo.cells[-1].source)

## 2.5 `generate_data_workflow` 

Everything combined with detect, filter, inject, write. The example `paleoPCAlite.ipynb` is
modified in place. Run the injected cell **in the demo's own kernel**, where
`filtered_df2` exists, to get the BibTeX.

Use `targets=` to request specific datasets by name.

In [ ]:
pairs = generate_data_workflow('paleoPCAlite.ipynb')
print('injected a retrieval cell for:', pairs)

targeted_pairs = generate_data_workflow(
    'paleoPCAlite.ipynb',
    tool='LiPDGraph',
    targets='Ocn-RedSea.Felis.2000',
)
print('injected a targeted retrieval cell for:', targeted_pairs)

---

# Part 3 - The agent layer

Parts 1 and 2 call the workflow functions directly. The agent wraps that same
work in three layers. They produce identical citations
and differ only in how you invoke them.

In [ ]:
DEMO = 'paleoPCAlite.ipynb'

## 3.1 The direct functions

`cite_software` returns the library names it built a cell for. `cite_data`
returns the `[variable, tool]` pairs it built a retrieval cell for. Neither
returns citations.

`cite_software` also takes the filters the natural-language layers expose through
language: `libraries=` for specific packages, `citation_types=` to keep only
`paper` or only `software` entries.

In [ ]:
from provenance_agent import cite_data, cite_software

print('software:', cite_software(DEMO))
print('software, filtered:', cite_software(
    DEMO,
    libraries=['pyleoclim', 'pandas'],
    citation_types=['software'],
))
print('data:', cite_data(DEMO))

## 3.2 The natural-language router - `agent.run`

Classifies the request into a typed decision, dispatches the workflows it
selected, and returns an envelope. **requires API key.

The envelope's `dispatch` key holds one `{name, args, result}` per tool called;
`verification` records what changed in the notebook.

An unclear request, or one naming software the notebook does not import, comes
back as `status: "warning"` with no changes.

In [ ]:
from provenance_agent.agent import run

result = run('cite the software', DEMO)

print('status:', result['status'])
for call in result['dispatch']:
    print(call['name'], '->', call['result'])
print('verification:', result['verification'])

## 3.3 The `%provenance` magic

In [ ]:
%load_ext provenance

Auto-detection matches the running kernel against the Jupyter server's current notebook.

In [ ]:
#set path manually
%provenance_notebook paleoPCAlite.ipynb

In [ ]:
%provenance cite the software

In [ ]:
%provenance cite the datasets

A named target can be defined

In [ ]:
%provenance cite Pyleoclim


## Next Steps

Reload `paleoPCAlite.ipynb` and run the cells the workflows
appended. You should see two cells.